### Step 1: Mount Google Drive and Install Dependencies
Access your dataset in Drive and install `yfinance` to fetch the stock data.

In [4]:
from google.colab import drive
import os
import pandas as pd
import numpy as np
import yfinance as yf
from tqdm import tqdm
from datetime import time
import pytz

# Mount Google Drive
drive.mount('/content/drive')

# Configuration Constants
BASE_PATH = '/content/drive/MyDrive/stock_dataset/'
IMAGE_DIR = os.path.join(BASE_PATH, 'raw_images/')
OUTPUT_CSV = os.path.join(BASE_PATH, 'labels.csv')
WINDOW_SIZE = 20
THRESHOLD = 0.005
TIMEZONE = 'Asia/Kolkata'
MARKET_START = time(9, 15)
MARKET_END = time(15, 30)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Step 2: Define Modular Functions
These functions handle data fetching, preprocessing, and the labeling logic.

In [5]:
def fetch_stock_data(ticker_symbol):
    """Fetch 60 days of 1h OHLC data for a given ticker."""
    ticker = yf.Ticker(ticker_symbol)
    df = ticker.history(period="60d", interval="1h")
    return df

def preprocess_data(df):
    """Convert timezone, filter market hours, and keep OHLC."""
    # Convert to Asia/Kolkata
    df.index = df.index.tz_convert(TIMEZONE)

    # Filter market hours
    df = df.between_time(MARKET_START, MARKET_END)

    # Select OHLC
    df = df[['Open', 'High', 'Low', 'Close']]
    return df

def get_label(return_val, threshold):
    """Assign label based on return value."""
    if return_val > threshold:
        return 'up'
    elif return_val < -threshold:
        return 'down'
    else:
        return 'neutral'

def process_stock_labels(stock_name, df, image_folder, window_size, threshold):
    """Generate labels for sliding windows aligned with image files."""
    results = []
    # Identify unique trading days
    days = sorted(df.index.normalize().unique())

    # We iterate through indices to create windows
    # Sliding window of window_size
    for i in range(len(df) - window_size):
        window = df.iloc[i : i + window_size]
        current_close = window.iloc[-1]['Close']
        current_time = window.index[-1]
        current_day = current_time.normalize()

        # Find index of current day in list of days
        try:
            day_idx = days.index(current_day)
            # Check if there is a next trading day available
            if day_idx + 1 < len(days):
                next_day = days[day_idx + 1]
                next_day_data = df[df.index.normalize() == next_day]

                if not next_day_data.empty:
                    next_day_close = next_day_data.iloc[-1]['Close']

                    # Calculate return
                    r = (next_day_close - current_close) / current_close
                    label = get_label(r, threshold)

                    # Match image path (assuming images are named image_0.png, image_1.png...)
                    img_path = os.path.join(image_folder, f"{stock_name}_{i}.png")

                    results.append({
                        "image_path": img_path,
                        "label": label,
                        "stock": stock_name,
                        "timestamp": current_time
                    })
        except ValueError:
            continue

    return results

### Step 3: Run the Pipeline
Iterate through all stock folders, process labels, and save the final CSV.

In [6]:
all_labels = []
stocks = [d for d in os.listdir(IMAGE_DIR) if os.path.isdir(os.path.join(IMAGE_DIR, d))]

print(f"Found {len(stocks)} stocks to process.")

for stock in tqdm(stocks, desc="Processing Stocks"):
    try:
        # 1. Fetch data
        raw_df = fetch_stock_data(stock + ".NS")
        if raw_df.empty:
            print(f"No data for {stock}, skipping.")
            continue

        # 2. Preprocess
        clean_df = preprocess_data(raw_df)

        # 3. Label windows
        stock_img_dir = os.path.join(IMAGE_DIR, stock)
        stock_results = process_stock_labels(stock, clean_df, stock_img_dir, WINDOW_SIZE, THRESHOLD)

        all_labels.extend(stock_results)
    except Exception as e:
        print(f"Error processing {stock}: {e}")

# 4. Create DataFrame
final_df = pd.DataFrame(all_labels)

# 5. Save to CSV
final_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nPipeline Complete. Labels saved to: {OUTPUT_CSV}")

# 6. Print Distribution
print("\n--- Label Distribution ---")
if final_df.empty:
    print("No labels generated. Check data fetching.")
else:
    print(final_df['label'].value_counts())
    print(final_df['label'].value_counts(normalize=True) * 100)
print("\n--- Class Balance (Percentage) ---")
print(final_df['label'].value_counts(normalize=True) * 100)

Found 50 stocks to process.


Processing Stocks: 100%|██████████| 50/50 [00:19<00:00,  2.60it/s]



Pipeline Complete. Labels saved to: /content/drive/MyDrive/stock_dataset/labels.csv

--- Label Distribution ---
label
down       8729
up         6458
neutral    3680
Name: count, dtype: int64
label
down       46.265967
up         34.229077
neutral    19.504956
Name: proportion, dtype: float64

--- Class Balance (Percentage) ---
label
down       46.265967
up         34.229077
neutral    19.504956
Name: proportion, dtype: float64
